# 01 — Common base (single cleaning and imputation step)

easyMoney · Cross-selling project

To keep every figure consistent across the project, **all cleaning and imputation is done only once, here**. The modelling branches start from this base:

```
Raw CSV files ──► 01_base_comun ──► df_panel_limpio.parquet ──┬─► 02  Task 2 (propensity)
                                    df_sales_limpio.parquet  ──┼─► 03  Task 3 (segmentation)
                                                               └─► 04  Task 4 (business case)
```

**What this notebook does:** clean, impute, join and save.

**What it does NOT do:** feature engineering. The `_lag1` features, `num_products`, `mes` (month) or `antigüedad` (tenure) are decisions for each task and are built inside its own branch. That way no task has to *undo* what another one did.

In [1]:
import os
import numpy as np
import pandas as pd

RUTA_RAW = '../data/raw/'
RUTA_OUT = '../data/processed/'
os.makedirs(RUTA_OUT, exist_ok=True)

df_sal = pd.read_csv(RUTA_RAW + 'sales.csv', index_col=0)
df_cs  = pd.read_csv(RUTA_RAW + 'customer_sociodemographics.csv', index_col=0)
df_cca = pd.read_csv(RUTA_RAW + 'customer_commercial_activity.csv', index_col=0)
df_cp  = pd.read_csv(RUTA_RAW + 'customer_products.csv', index_col=0)

for nombre, d in [('sales', df_sal), ('sociodemographics', df_cs),
                  ('commercial_activity', df_cca), ('products', df_cp)]:
    print(f'{nombre:20s} {d.shape}')

sales                (240773, 5)
sociodemographics    (5962924, 8)
commercial_activity  (5962924, 6)
products             (5962924, 17)


## 1. Products (`customer_products`)

Only `payroll` and `pension_plan` have missing values (61 rows each), but we apply the same rule to all 15 columns just in case.

In [2]:
PRODUCT_COLS = ['short_term_deposit', 'loans', 'mortgage', 'funds', 'securities',
                'long_term_deposit', 'em_account_pp', 'credit_card', 'payroll',
                'pension_plan', 'payroll_account', 'emc_account', 'debit_card',
                'em_account_p', 'em_acount']

print('Missing before:', df_cp[PRODUCT_COLS].isnull().sum()[lambda s: s > 0].to_dict())
df_cp[PRODUCT_COLS] = df_cp[PRODUCT_COLS].fillna(0).astype(int)
print('Missing after:', df_cp[PRODUCT_COLS].isnull().sum().sum())

Missing before: {'payroll': 61, 'pension_plan': 61}
Missing after: 0


## 2. Removing deceased customers

We remove the **months** in which the customer is flagged as deceased. Since we later do an `inner` join with this table, those months also disappear from the panel and therefore from tasks 2, 3 and 4.

In [3]:
print(df_cs['deceased'].value_counts(dropna=False))
filas_antes = len(df_cs)
df_cs = df_cs[df_cs['deceased'] == 'N'].drop(columns=['deceased'])
print(f'\nRows removed (deceased): {filas_antes - len(df_cs)}')

deceased
N    5961849
S       1075
Name: count, dtype: int64

Rows removed (deceased): 1075


## 3. Building the customer-month panel

We join the three tables on `pk_cid` + `pk_partition`:
- `inner` with sociodemographics: this way we inherit the deceased filter.
- `left` with commercial activity: nothing has been filtered in that table.

We sort by customer and date. Any later `ffill`, `bfill` or `shift` depends on this order; if it were not sorted, the results would be wrong without raising any error.

In [4]:
for d in (df_cp, df_cs, df_cca):
    d['pk_partition'] = pd.to_datetime(d['pk_partition'], format='%Y-%m')
df_cca['entry_date'] = pd.to_datetime(df_cca['entry_date'], format='%Y-%m')

df_panel = (df_cp
            .merge(df_cs, on=['pk_cid', 'pk_partition'], how='inner')
            .merge(df_cca, on=['pk_cid', 'pk_partition'], how='left')
            .sort_values(['pk_cid', 'pk_partition'])
            .reset_index(drop=True))

print('Panel shape:', df_panel.shape)
print('Unique customers:', df_panel['pk_cid'].nunique())
print('Months:', df_panel['pk_partition'].min().strftime('%Y-%m'), '->',
      df_panel['pk_partition'].max().strftime('%Y-%m'))
print('\nMissing values after the merge:')
print(df_panel.isnull().sum()[lambda s: s > 0])

Panel shape: (5961849, 26)
Unique customers: 456318
Months: 2018-01 -> 2019-05

Missing values after the merge:
region_code         2259
gender                25
salary           1540876
entry_channel     133033
segment           133944
dtype: int64


## 4. Imputation from the customer's own history

`gender`, `region_code`, `entry_channel` and `segment` are customer attributes: if one is missing in a given month, the most reliable value is the one the same customer has in other months.

- First `ffill`: the last known value.
- Then `bfill`: for missing values in the customer's first months.

**About `bfill` and leakage:** filling a month with a value from a later month uses future information. We accept it here because these are attributes assigned at sign-up (entry channel, segment, province, gender), not behaviour. They say nothing about whether the customer will buy a product. We document it in case the examination panel asks.

Whatever is still missing after this (customers who **never** have the value) goes to the rules in the next section.

In [5]:
COLS_HISTORIA = ['gender', 'region_code', 'entry_channel', 'segment']

nulos_antes = df_panel[COLS_HISTORIA].isnull().sum()
df_panel[COLS_HISTORIA] = df_panel.groupby('pk_cid')[COLS_HISTORIA].ffill()
df_panel[COLS_HISTORIA] = df_panel.groupby('pk_cid')[COLS_HISTORIA].bfill()
nulos_despues = df_panel[COLS_HISTORIA].isnull().sum()

print(pd.DataFrame({'missing_before': nulos_antes,
                    'filled_from_history': nulos_antes - nulos_despues,
                    'remaining': nulos_despues}))

               missing_before  filled_from_history  remaining
gender                     25                    0         25
region_code              2259                  126       2133
entry_channel          133033               123698       9335
segment                133944               124876       9068


## 5. What the history does not solve

- **`gender`:** global mode. Affects very few rows.
- **`region_code`:** mode for Spain for customers with `country_id == 'ES'`, `0` for foreign customers. That is why `country_id` is **not** dropped before imputing.
- **`entry_channel` / `segment`:** cross imputation. If the channel is missing but the segment is present, we use the mode of the channel within that segment, and vice versa. If both are missing, `"Desconocido"` (Unknown).

In [6]:
# gender
df_panel['gender'] = df_panel['gender'].fillna(df_panel['gender'].mode()[0])

# region_code conditional on country_id
moda_region_es = df_panel.loc[df_panel['country_id'] == 'ES', 'region_code'].mode()[0]
mask_es  = df_panel['region_code'].isnull() & (df_panel['country_id'] == 'ES')
mask_ext = df_panel['region_code'].isnull() & (df_panel['country_id'] != 'ES')
df_panel.loc[mask_es,  'region_code'] = moda_region_es
df_panel.loc[mask_ext, 'region_code'] = 0
df_panel['region_code'] = df_panel['region_code'].astype(int)
print(f'region_code -> ES mode ({moda_region_es:.0f}): {mask_es.sum()} | foreign (0): {mask_ext.sum()}')

# entry_channel / segment: cross imputation
def moda_segura(x):
    m = x.mode()
    return m.iloc[0] if not m.empty else np.nan

mask_canal = df_panel['entry_channel'].isnull() & df_panel['segment'].notnull()
mask_seg   = df_panel['segment'].isnull() & df_panel['entry_channel'].notnull()

moda_canal_por_seg = df_panel.groupby('segment')['entry_channel'].agg(moda_segura)
moda_seg_por_canal = df_panel.groupby('entry_channel')['segment'].agg(moda_segura)

df_panel.loc[mask_canal, 'entry_channel'] = df_panel.loc[mask_canal, 'segment'].map(moda_canal_por_seg)
df_panel.loc[mask_seg,   'segment']       = df_panel.loc[mask_seg, 'entry_channel'].map(moda_seg_por_canal)

mask_ambos = df_panel['entry_channel'].isnull() | df_panel['segment'].isnull()
df_panel['entry_channel'] = df_panel['entry_channel'].fillna('Desconocido')
df_panel['segment'] = df_panel['segment'].fillna('Desconocido')
print(f'Channel from segment: {mask_canal.sum()} | Segment from channel: {mask_seg.sum()} | Unknown (Desconocido): {mask_ambos.sum()}')

# active_customer
df_panel['active_customer'] = df_panel['active_customer'].fillna(0).astype(int)

region_code -> ES mode (28): 0 | foreign (0): 2133
Channel from segment: 612 | Segment from channel: 345 | Unknown (Desconocido): 8724


## 6. `salary`: imputation

`salary` is the variable with the most missing values. It is imputed in four steps, from most to least reliable:

1. **Flag the original missing values** with `salary_imputado` (1 = estimated, 0 = actual), before touching anything.
2. **The customer's own history:** if it is missing in one month but present in another, the customer's own value is copied.
3. **See what salary depends on** to choose which variables to group by.
4. **Median of similar customers** for those with no salary in any month.

We use the **median** rather than the mean because `salary` has extreme values that would inflate the mean.

In [7]:
# Step 1: flag the original missing values (before imputing anything)
df_panel['salary_imputado'] = df_panel['salary'].isnull().astype(int)
print(f"Originally missing salary: {df_panel['salary_imputado'].sum():,} rows "
      f"({df_panel['salary_imputado'].mean():.1%} of the panel)")

# Step 2: the customer's own history (forward and backward)
df_panel = df_panel.sort_values(['pk_cid', 'pk_partition'])
df_panel['salary'] = df_panel.groupby('pk_cid')['salary'].transform(lambda s: s.ffill().bfill())

pendientes = df_panel['salary'].isnull().sum()
print(f"Recovered from the customer's history: {df_panel['salary_imputado'].sum() - pendientes:,}")
print(f"Still missing salary (the customer has no value in any month): {pendientes:,} rows "
      f"| {df_panel.loc[df_panel['salary'].isnull(), 'pk_cid'].nunique():,} customers")

Originally missing salary: 1,540,876 rows (25.8% of the panel)
Recovered from the customer's history: 621
Still missing salary (the customer has no value in any month): 1,540,255 rows | 158,752 customers


### Step 3: what does salary depend on?

For each candidate variable we look at how much the median salary changes across its categories, and what share of salary variation it explains (**η²**, from 0 to 1: the higher, the more the variable explains).

It is computed on the **logarithm** of salary so that extreme salaries do not dominate, and only with actual values (`salary_imputado == 0`).

In [8]:
df_panel['tramo_edad'] = pd.cut(df_panel['age'], bins=[0, 25, 35, 45, 55, 65, 120],
                                labels=['<25', '25-34', '35-44', '45-54', '55-64', '65+'])

reales = df_panel[df_panel['salary_imputado'] == 0].copy()
reales['log_salary'] = np.log1p(reales['salary'])
var_total = ((reales['log_salary'] - reales['log_salary'].mean()) ** 2).sum()

filas = []
for col in ['region_code', 'segment', 'tramo_edad', 'gender', 'entry_channel']:
    g = reales.groupby(col, observed=True)['log_salary'].agg(['mean', 'count'])
    var_entre = (g['count'] * (g['mean'] - reales['log_salary'].mean()) ** 2).sum()
    med = reales.groupby(col, observed=True)['salary'].median()
    filas.append({'variable': col, 'categorias': len(g),
                  'mediana_min': round(med.min()), 'mediana_max': round(med.max()),
                  'eta2': round(var_entre / var_total, 3)})

relacion_salary = pd.DataFrame(filas).sort_values('eta2', ascending=False)
print(relacion_salary.to_string(index=False))

     variable  categorias  mediana_min  mediana_max  eta2
  region_code          52        53029       126179 0.189
entry_channel          61        32627       402884 0.007
      segment           4        87851       121099 0.004
   tramo_edad           6        85370       102747 0.004
       gender           2        87201        89909 0.001


### Step 4: median of similar customers

Each row without a salary gets the median of customers with the same combination of variables (`VARIABLES_SALARY`, from most to least important).

If the combination has fewer than `MIN_GRUPO` customers with an actual salary, the median is not reliable, so we **go down one level** by dropping the last variable in the list, and so on until we reach the global median. For example, with `['region_code', 'segment', 'tramo_edad']` (age band):

1. region × segment × age band
2. if there are not enough → region × segment
3. if not → region
4. if not → global median

Medians are computed only with **actual** salaries, never with already imputed values.

In [9]:
VARIABLES_SALARY = ['region_code', 'segment', 'tramo_edad']   # check against the table in step 3
MIN_GRUPO = 30                                                 # minimum customers to trust a median

# One row per customer with an actual salary (so a customer with 17 months doesn't count 17 times)
base_medianas = (df_panel[df_panel['salary_imputado'] == 0]
                 .groupby('pk_cid').last()[VARIABLES_SALARY + ['salary']])

faltan = df_panel['salary'].isnull()
origen = pd.Series('real or history', index=df_panel.index)

for n in range(len(VARIABLES_SALARY), 0, -1):
    claves = VARIABLES_SALARY[:n]
    stats = base_medianas.groupby(claves, observed=True)['salary'].agg(['median', 'count'])
    stats = stats[stats['count'] >= MIN_GRUPO]['median'].rename('mediana_grupo').reset_index()

    pendientes = (df_panel.loc[faltan, claves].rename_axis('fila').reset_index()
                  .merge(stats, on=claves, how='left').set_index('fila'))
    rellenar = pendientes['mediana_grupo'].dropna()
    df_panel.loc[rellenar.index, 'salary'] = rellenar
    origen.loc[rellenar.index] = ' × '.join(claves)
    faltan = df_panel['salary'].isnull()

mediana_global = base_medianas['salary'].median()
origen.loc[faltan] = 'global median'
df_panel['salary'] = df_panel['salary'].fillna(mediana_global)

print('How each row with originally missing salary was imputed:')
print(origen[df_panel['salary_imputado'] == 1].value_counts().to_string())
print(f"\nMissing salary after imputation: {df_panel['salary'].isnull().sum()}")

df_panel = df_panel.drop(columns=['tramo_edad'])

How each row with originally missing salary was imputed:
region_code × segment × tramo_edad    1415930
global median                          100207
region_code × segment                   16305
region_code                              7813
real or history                           621

Missing salary after imputation: 0


In [10]:
# Check: imputation should not distort the salary distribution
comparacion = pd.DataFrame({
    'actual':   df_panel.loc[df_panel['salary_imputado'] == 0, 'salary'].describe(),
    'imputed':  df_panel.loc[df_panel['salary_imputado'] == 1, 'salary'].describe(),
}).round(0)
print(comparacion)

           actual    imputed
count   4420973.0  1540876.0
mean     115581.0    91920.0
std      200018.0    24959.0
min        1203.0    27669.0
25%       61414.0    70120.0
50%       88438.0    87993.0
75%      131308.0   108593.0
max    28894396.0   470033.0


**Note.** Medians are computed using all months (January 2018 – May 2019). This is a cleaning decision for the common base: a customer's imputed value is a stable characteristic (their salary), not something that changes month to month.

## 7. Sales (`sales`)

1. `month_sale` is converted to a monthly date, with the same format as `pk_partition`.
2. We keep only sales from customer-months that exist in the panel. This also excludes sales from months of deceased customers.
3. Two outputs are saved:
   - **Clean sales** (one row per sale) for Task 4, which needs the margin per product.
   - **Customer-month aggregate** (`n_compras_mes` = purchases in the month, `margen_mes` = margin in the month) joined to the panel, for Task 3.

In [11]:
df_sal['pk_partition'] = pd.to_datetime(df_sal['month_sale']).dt.to_period('M').dt.to_timestamp()
df_sal = df_sal.rename(columns={'cid': 'pk_cid'}).drop(columns=['month_sale'])

claves_panel = df_panel[['pk_cid', 'pk_partition']]
ventas_antes = len(df_sal)
df_sales_limpio = df_sal.merge(claves_panel, on=['pk_cid', 'pk_partition'], how='inner')
print(f'Total sales: {ventas_antes} | in the panel: {len(df_sales_limpio)} | '
      f'discarded (customer-month outside the panel): {ventas_antes - len(df_sales_limpio)}')

ventas_mes = (df_sales_limpio
              .groupby(['pk_cid', 'pk_partition'])
              .agg(n_compras_mes=('pk_sale', 'count'), margen_mes=('net_margin', 'sum'))
              .reset_index())

df_panel = df_panel.merge(ventas_mes, on=['pk_cid', 'pk_partition'], how='left')
df_panel[['n_compras_mes', 'margen_mes']] = df_panel[['n_compras_mes', 'margen_mes']].fillna(0)
df_panel['n_compras_mes'] = df_panel['n_compras_mes'].astype(int)

# Check: the panel aggregate matches the clean sales
assert df_panel['n_compras_mes'].sum() == len(df_sales_limpio)
assert np.isclose(df_panel['margen_mes'].sum(), df_sales_limpio['net_margin'].sum())
print('Sales aggregate matches the clean sales.')

Total sales: 240773 | in the panel: 240752 | discarded (customer-month outside the panel): 21
Sales aggregate matches the clean sales.


## 8. Final checks and saving

In [12]:
df_panel = df_panel.sort_values(['pk_cid', 'pk_partition']).reset_index(drop=True)

nulos = df_panel.isnull().sum()
print('Columns with missing values (there should be none):')
print(nulos[nulos > 0] if nulos.any() else 'None')
assert not nulos.any(), 'Unexpected missing values'
assert not df_panel.duplicated(['pk_cid', 'pk_partition']).any(), 'Duplicated customer-months'

print('\nFinal shape:', df_panel.shape)
print('Columns:', list(df_panel.columns))

Columns with missing values (there should be none):
None

Final shape: (5961849, 29)
Columns: ['pk_cid', 'pk_partition', 'short_term_deposit', 'loans', 'mortgage', 'funds', 'securities', 'long_term_deposit', 'em_account_pp', 'credit_card', 'payroll', 'pension_plan', 'payroll_account', 'emc_account', 'debit_card', 'em_account_p', 'em_acount', 'country_id', 'region_code', 'gender', 'age', 'salary', 'entry_date', 'entry_channel', 'active_customer', 'segment', 'salary_imputado', 'n_compras_mes', 'margen_mes']


In [13]:
df_panel.to_parquet(RUTA_OUT + 'df_panel_limpio.parquet', engine='pyarrow', index=False)
df_sales_limpio.to_parquet(RUTA_OUT + 'df_sales_limpio.parquet', engine='pyarrow', index=False)
print('Saved:')
print(' -', RUTA_OUT + 'df_panel_limpio.parquet', df_panel.shape)
print(' -', RUTA_OUT + 'df_sales_limpio.parquet', df_sales_limpio.shape)

Saved:
 - ../data/processed/df_panel_limpio.parquet (5961849, 29)
 - ../data/processed/df_sales_limpio.parquet (240752, 5)


## How each task uses it

- **`02_tarea2_propension`:** creates `num_products`, `mes`, `antigüedad`, the `_lag1` features and the targets, and compares 4 models for `pension_plan` and `em_acount`.
- **`03_tarea3_segmentacion`:** customer snapshot + sales history.
- **Task 4:** uses `df_sales_limpio.parquet` for the margin per product, plus the Task 2 scorings and the Task 3 segments. All three come from the same customer universe.